# Hockey Analytics: Phase 1 Data Cleaning & Sequence Preparation

**Objective**: Clean and prepare raw Parquet event data by:
1. Removing shootouts and penalty shots
2. Validating event sequence order
3. Handling assists and goal leakage
4. Sorting events causally within sequences
5. Preparing data for tensor conversion

**Output**: A cleaned, tensor-ready parquet file ready for Phase 2 sequence assembly

## 1. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print("✓ Libraries imported successfully")

# Set base directory to parent of current directory
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "HALO Hackathon Data"
OUTPUT_DIR = BASE_DIR / "Processed Sequence Data"

# Create output directory if it doesn't exist
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Base Directory: {BASE_DIR}")
print(f"Data Directory: {DATA_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")

✓ Libraries imported successfully
Base Directory: x:\My Files\Python\Sports Analytics Projects\Hockey\HALO
Data Directory: x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\HALO Hackathon Data
Output Directory: x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\Processed Sequence Data


## 2. Load Raw Parquet Data

In [2]:
# Use DATA_DIR from the first cell
data_dir = DATA_DIR

# Check available parquet files
parquet_files = list(data_dir.glob("*.parquet"))
print(f"Found {len(parquet_files)} parquet files in {data_dir}:")
for f in parquet_files[:10]:  # Show first 10
    print(f"  - {f.name}")

# Assuming the main events file is present
# For now, we'll check what's available
if parquet_files:
    # Load the first/main parquet file as events
    events_path = parquet_files[0]
else:
    # Default expected path
    events_path = data_dir / "events.parquet"

print(f"\nLoading: {events_path}")
try:
    df_events = pd.read_parquet(events_path)
    print(f"✓ Loaded {len(df_events):,} events")
    print(f"✓ Shape: {df_events.shape}")
    print(f"\nColumn names and types:")
    print(df_events.dtypes)
except Exception as e:
    print(f"✗ Error loading parquet: {e}")
    print("Creating sample DataFrame for exploration...")
    df_events = None

Found 5 parquet files in x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\HALO Hackathon Data:
  - events.parquet
  - games.parquet
  - players.parquet
  - stints.parquet
  - tracking.parquet

Loading: x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\HALO Hackathon Data\events.parquet
✓ Loaded 1,800,464 events
✓ Shape: (1800464, 24)

Column names and types:
game_id                  object
period                    int64
period_time              object
game_stint              float64
sl_event_id               int64
sequence_id               int64
player_id                object
player_name              object
team                     object
team_id                  object
opp_team                 object
opp_team_id              object
event_type               object
outcome                  object
flags                    object
description              object
detail                   object
sl_xg_all_shots         float64
x                       float64
y               

## 3. Explore Data Structure

In [3]:
if df_events is not None:
    print("=== Basic Info ===")
    print(f"Total events: {len(df_events):,}")
    print(f"Date range: {df_events.get('game_date', df_events.iloc[:, 0]).min()} to {df_events.get('game_date', df_events.iloc[:, 0]).max()}")
    
    print("\n=== Missing Values ===")
    missing = df_events.isnull().sum()
    if missing.sum() > 0:
        print(missing[missing > 0])
    else:
        print("No missing values")
    
    print("\n=== Sample Rows ===")
    print(df_events.head(10))
    
    # Check for key columns
    key_cols = ['game_id', 'sequence_id', 'event_type', 'period', 'period_time', 'team_id', 'player_id']
    available_cols = [col for col in key_cols if col in df_events.columns]
    missing_cols = [col for col in key_cols if col not in df_events.columns]
    
    print(f"\n=== Key Columns ===")
    print(f"Available: {available_cols}")
    if missing_cols:
        print(f"Missing: {missing_cols}")
    
    # Event type distribution
    if 'event_type' in df_events.columns:
        print(f"\n=== Event Type Distribution ===")
        event_counts = df_events['event_type'].value_counts()
        print(event_counts)
    
    # Period distribution
    if 'period' in df_events.columns:
        print(f"\n=== Period Distribution ===")
        print(df_events['period'].value_counts().sort_index())

=== Basic Info ===
Total events: 1,800,464
Date range: 00b0366a-95c6-5250-2dae-e3dd5c4198bc to ff8386b6-5f9b-4734-00e6-bc8b4c0a4cca

=== Missing Values ===
game_stint            1053
player_id            64310
player_name          64310
team                 64310
team_id              64310
opp_team             64310
opp_team_id          64310
outcome              64310
flags               990106
sl_xg_all_shots    1746484
x                    64310
y                    64310
x_adj                64310
y_adj                64310
dtype: int64

=== Sample Rows ===
                                game_id  period  period_time  game_stint  \
0  00b0366a-95c6-5250-2dae-e3dd5c4198bc       1         0E-9         1.0   
1  00b0366a-95c6-5250-2dae-e3dd5c4198bc       1         0E-9         1.0   
2  00b0366a-95c6-5250-2dae-e3dd5c4198bc       1         0E-9         1.0   
3  00b0366a-95c6-5250-2dae-e3dd5c4198bc       1  0.070000000         1.0   
4  00b0366a-95c6-5250-2dae-e3dd5c4198bc       1  0.1

## 4. Remove Shootouts and Penalty Shots

Per Phase 1 specifications, remove all shootout and penalty shot events:
- `shootout`, `teamwithozonright`, `solpr`, `socarry`, `soshot`, `sogoal`, `sopuckprotection`, `socheck`

In [4]:
# Preview: events that remain after any shootout/penaltyshot activity
print("\n=== 4.0: Inspect Post-Shootout Activity ===")
shootout_related = {
    'shootout', 'solpr', 'socarry', 'soshot',
    'sogoal', 'sopuckprotection', 'socheck'
}

penalty_shot_related = {
    'penaltyshot', 'pslpr', 'pscarry', 'teamwithozonright'
}
shootout_mask = df_events['event_type'].isin(shootout_related)

if shootout_mask.any():
    earliest = df_events[shootout_mask].sort_values(['period', 'period_time']).head(1)
    print("Earliest shootout-related event:")
    print(earliest[['game_id', 'period', 'period_time', 'sequence_id', 'event_type']])
else:
    print("No shootout-related events found")

remaining_events = []

for game_id, game_df in df_events.groupby('game_id'):
    triggers = game_df[game_df['event_type'].isin(shootout_related)]
    if triggers.empty:
        continue

    first_trigger = triggers.index.min()
    tail = df_events.loc[first_trigger + 1:]
    tail = tail[tail['game_id'] == game_id]
    if not tail.empty:
        remaining_events.append(tail[['game_id', 'period', 'period_time', 'sequence_id', 'event_type']])

if remaining_events:
    after_df = pd.concat(remaining_events)
    print(f"Found {len(after_df):,} events after the first shootout-related event (per game)")
    print("Event counts:")
    print(after_df['event_type'].value_counts().head(10))
    print("\nSample tail events:")
    print(after_df.groupby(['game_id', 'event_type']).head(3).reset_index(drop=True))
else:
    print("No events occur after shootout-related activity")

penaltyshot_windows = []
checksum = df_events['event_type'] == 'penaltyshot'
for idx in df_events[checksum].index:
    start_seq = df_events.loc[idx, 'sequence_id']
    game_id = df_events.loc[idx, 'game_id']
    window = []
    j = idx
    while j < len(df_events):
        row = df_events.loc[j]
        if row['game_id'] != game_id:
            break
        window.append(row[['event_type', 'sequence_id', 'period_time']])
        if row['event_type'] == 'faceoff' and row['sequence_id'] != start_seq:
            break
        j += 1
    penaltyshot_windows.append(pd.DataFrame(window))

print(f"\nDetected {len(penaltyshot_windows)} penaltyshot windows")
if penaltyshot_windows:
    merged = pd.concat(penaltyshot_windows, keys=range(len(penaltyshot_windows)))
    print(merged.groupby(['event_type']).size())
    print("Sample penaltyshot window (first 5 rows):")
    print(penaltyshot_windows[0].reset_index(drop=True).head())
else:
    print("No penaltyshot sequences found")


=== 4.0: Inspect Post-Shootout Activity ===
Earliest shootout-related event:
                                    game_id  period    period_time  \
30180  06795c32-cffd-e9e8-3a6a-5728a7f07a8b       4  299.970000000   

       sequence_id event_type  
30180           65   shootout  
Found 898 events after the first shootout-related event (per game)
Event counts:
event_type
solpr                237
socarry              237
soshot               231
sogoal                76
teamwithozonright     39
shootout              35
goal                  35
sopuckprotection       7
socheck                1
Name: count, dtype: int64

Sample tail events:
                                  game_id  period    period_time  sequence_id  \
0    00f1ee7c-b2e4-3fee-b8ba-37158dc3166d       4  300.000000000           53   
1    00f1ee7c-b2e4-3fee-b8ba-37158dc3166d       4  300.000000000           53   
2    00f1ee7c-b2e4-3fee-b8ba-37158dc3166d       4  300.000000000           53   
3    00f1ee7c-b2e4-3fee-b8ba-

### 4.0.1 Extract Goals from Penalty Shots & Shootouts

**Before removal**, extract penalty shot goals and shootout winners for score verification:

1. **Penalty Shot Goals**: Track successful penalty shots (goal scored)
2. **Shootout Winners**: Determine which team won the shootout by counting goals

These will be saved to `extra_goals_for_verification.parquet` and used later to verify that our cleaned event data produces correct game scores when compared to the official `games.parquet` file.

In [5]:
print("=== 4.0.1: Extract Penalty Shot & Shootout Goals for Score Verification ===")
print("Extracting goals from penalty shots and shootouts BEFORE removal")
print("Note: Only counting goals with player_id to avoid duplicate goal events\n")

# ---------------------------------------------------------
# Part 1: Extract Penalty Shot Goals
# ---------------------------------------------------------
print("--- Part 1: Penalty Shot Goals ---")

penalty_shot_goals = []

# Find all penalty shot sequences
penaltyshot_indices = df_events[df_events['event_type'] == 'penaltyshot'].index.tolist()
print(f"Found {len(penaltyshot_indices)} penalty shot sequences")

for ps_idx in penaltyshot_indices:
    ps_row = df_events.loc[ps_idx]
    game_id = ps_row['game_id']
    seq_id = ps_row['sequence_id']
    
    # Get all events in this penalty shot sequence (until next sequence or game change)
    ps_sequence = df_events[
        (df_events['game_id'] == game_id) &
        (df_events['sequence_id'] == seq_id) &
        (df_events.index >= ps_idx)
    ]
    
    # Check if there's a goal in this sequence (with player_id to avoid duplicates)
    goals_in_sequence = ps_sequence[
        (ps_sequence['event_type'] == 'goal') &
        (ps_sequence['player_id'].notna())
    ]
    
    if len(goals_in_sequence) > 0:
        # Penalty shot resulted in a goal
        goal_row = goals_in_sequence.iloc[0]
        penalty_shot_goals.append({
            'game_id': game_id,
            'team_id': goal_row['team_id'],
            'period': goal_row['period'],
            'period_time': float(goal_row['period_time']),
            'goal_type': 'penalty_shot'
        })

df_penalty_shot_goals = pd.DataFrame(penalty_shot_goals)
print(f"✓ Found {len(df_penalty_shot_goals)} penalty shot goals")
if len(df_penalty_shot_goals) > 0:
    print(f"  Games with penalty shot goals: {df_penalty_shot_goals['game_id'].nunique()}")
    print(f"  Sample:")
    print(df_penalty_shot_goals.head(3))

# ---------------------------------------------------------
# Part 2: Extract Shootout Winners
# ---------------------------------------------------------
print("\n--- Part 2: Shootout Winners ---")

shootout_winners = []

# Find games with shootout events (period 4, time > 299)
shootout_games = df_events[
    (df_events['period'] == 4) & 
    (df_events['period_time'] > 299) &
    (df_events['event_type'].isin(['shootout', 'sogoal']))
]['game_id'].unique()

print(f"Found {len(shootout_games)} games with shootouts")

for game_id in shootout_games:
    # Get all events after period 4, time 299 for this game
    shootout_events = df_events[
        (df_events['game_id'] == game_id) &
        (df_events['period'] == 4) &
        (df_events['period_time'] > 299)
    ]
    
    # Count shootout goals by team (only goals with player_id to avoid duplicates)
    shootout_goals = shootout_events[
        (shootout_events['event_type'] == 'sogoal') &
        (shootout_events['player_id'].notna())
    ]
    
    if len(shootout_goals) > 0:
        goal_counts = shootout_goals['team_id'].value_counts()
        
        if len(goal_counts) > 0:
            # Winner is the team with more goals
            winning_team = goal_counts.idxmax()
            winning_goals = goal_counts.max()
            
            # Check for tie (shouldn't happen in shootouts, but just in case)
            if len(goal_counts) > 1 and goal_counts.iloc[0] == goal_counts.iloc[1]:
                print(f"  ⚠ Warning: Game {game_id[:8]}... has tied shootout ({goal_counts.iloc[0]} each)")
                continue
            
            shootout_winners.append({
                'game_id': game_id,
                'team_id': winning_team,
                'shootout_goals': winning_goals,
                'goal_type': 'shootout_winner'
            })

df_shootout_winners = pd.DataFrame(shootout_winners)
print(f"✓ Found {len(df_shootout_winners)} shootout winners")
if len(df_shootout_winners) > 0:
    print(f"  Sample:")
    print(df_shootout_winners.head(3))

# ---------------------------------------------------------
# Part 3: Combine and Save for Later Verification
# ---------------------------------------------------------
print("\n--- Part 3: Save for Score Verification ---")

# Combine both DataFrames
extra_goals = pd.concat([df_penalty_shot_goals, df_shootout_winners], ignore_index=True)

print(f"\n✓ Total extra goals to track: {len(extra_goals)}")
print(f"  - Penalty shot goals: {len(df_penalty_shot_goals)}")
print(f"  - Shootout winners: {len(df_shootout_winners)}")

# Save to output directory for later verification
output_path_extra_goals = OUTPUT_DIR / "extra_goals_for_verification.csv"
extra_goals.to_csv(output_path_extra_goals, index=False)
print(f"\n✓ Saved to {output_path_extra_goals.name}")
print("  (These goals will be used later to verify cleaned data matches game scores)")

print("\n" + "="*70)

=== 4.0.1: Extract Penalty Shot & Shootout Goals for Score Verification ===
Extracting goals from penalty shots and shootouts BEFORE removal
Note: Only counting goals with player_id to avoid duplicate goal events

--- Part 1: Penalty Shot Goals ---
Found 40 penalty shot sequences
✓ Found 6 penalty shot goals
  Games with penalty shot goals: 6
  Sample:
                                game_id                               team_id  \
0  00b0366a-95c6-5250-2dae-e3dd5c4198bc  d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019   
1  03470158-c116-5442-a415-c1b51c5ad0c3  c1cd64c9-e2fd-7a1a-269a-aa5a8e422cbb   
2  0df3ba0c-8682-c253-27fa-0c5efecfdcb2  f630ab4e-2bc7-6672-9c9c-4a4c835b198c   

   period  period_time     goal_type  
0       4        86.03  penalty_shot  
1       1       192.87  penalty_shot  
2       1       653.17  penalty_shot  

--- Part 2: Shootout Winners ---
Found 35 games with shootouts
✓ Found 35 shootout winners
  Sample:
                                game_id                       

In [6]:
# Phase 1 Cleanup: Remove shootout and penalty shot context events
print("=== Phase 1: Remove Shootout & Penalty Shot Context Events ===")
print(f"Starting with {len(df_events):,} events")

# Remove all events where period = 4 and time >= 300 (shootout events)
shootout_mask = (df_events['period'] == 4) & (df_events['period_time'] >= 300)
shootout_count = shootout_mask.sum()
print(f"✓ Removing {shootout_count:,} shootout events (period=4 & time>=300)")
df_events = df_events[~shootout_mask].reset_index(drop=True)
print(f"✓ Remaining events after shootout removal: {len(df_events):,}")   

# Remove everything that occurs after the first 'shootout' event in each game (period 4, time>299)
post_event_indices = []
shootout_trigger = df_events[
    (df_events['event_type'] == 'shootout') &
    (df_events['period'] == 4) &
    (df_events['period_time'] > 299)
]
for game_id, trigger_group in shootout_trigger.groupby('game_id'):
    first_idx = trigger_group.index.min()
    tail = df_events.loc[first_idx + 1:]
    tail = tail[tail['game_id'] == game_id]
    post_event_indices.extend(tail.index.tolist())

# Remove sequences that start with a penaltyshot and continue until the next sequence-changing faceoff
penaltyshot_indices = []
penaltyshot_mask = df_events['event_type'] == 'penaltyshot'
for idx in df_events[penaltyshot_mask].index:
    start_seq = df_events.loc[idx, 'sequence_id']
    game_id = df_events.loc[idx, 'game_id']
    j = idx
    while j < len(df_events):
        row = df_events.loc[j]
        if row['game_id'] != game_id:
            break
        if row['event_type'] == 'faceoff' and row['sequence_id'] != start_seq:
            break
        penaltyshot_indices.append(j)
        j += 1

suppress_indices = set(post_event_indices) | set(penaltyshot_indices)
if suppress_indices:
    print(f"\n✓ Removing {len(suppress_indices):,} shootout/penaltyshot-tail events")
    df_events = df_events.drop(index=suppress_indices).reset_index(drop=True)
    print(f"✓ Events remaining after tail removal: {len(df_events):,}")
else:
    print("\n✓ No extra shootout or penaltyshot tail events found")

# Event types to remove (shootout-specific events)
event_types_to_remove = [
    'shootout',          # Shootout events
    'teamwithozonright', # Team with OZ on right (shootout context)
    'solpr',             # Shootout LPR
    'socarry',           # Shootout carry
    'soshot',            # Shootout shot
    'sogoal',            # Shootout goal
    'sopuckprotection',  # Shootout puck protection
    'socheck'            # Shootout check
]

# Count events being removed
remove_mask = df_events['event_type'].isin(event_types_to_remove)
removal_counts = df_events[remove_mask]['event_type'].value_counts()
print(f"\nEvent types being removed:")
print(removal_counts)

# Remove them
df_events = df_events[~remove_mask].reset_index(drop=True)
print(f"\n✓ Removed {remove_mask.sum():,} events")
print(f"✓ Remaining events: {len(df_events):,}")

# Keep penaltyshot events (they are sequence terminators) 
ps_count = (df_events['event_type'] == 'penaltyshot').sum()
print(f"✓ Retained {ps_count:,} penaltyshot events")

print("\n=== Updated Event Type Distribution ===")
print(df_events['event_type'].value_counts())

=== Phase 1: Remove Shootout & Penalty Shot Context Events ===
Starting with 1,800,464 events
✓ Removing 863 shootout events (period=4 & time>=300)
✓ Remaining events after shootout removal: 1,799,601

✓ Removing 223 shootout/penaltyshot-tail events
✓ Events remaining after tail removal: 1,799,378

Event types being removed:
event_type
shootout    4
Name: count, dtype: int64

✓ Removed 4 events
✓ Remaining events: 1,799,374
✓ Retained 0 penaltyshot events

=== Updated Event Type Distribution ===
event_type
pass                      402670
lpr                       338836
reception                 304903
carry                     110362
failedpasslocation         97366
faceoff                    84621
block                      66233
puckprotection             54827
pressure                   53961
shot                       53952
controlledentryagainst     38081
dumpout                    35152
dumpin                     34740
dumpinagainst              32182
whistle                   

In [7]:
# Remove penalty bookkeeping events (start/end markers) only when unassigned
print("=== 4.1: Remove Unassigned Penalty Bookkeeping Events ===")
print(f"Starting with {len(df_events):,} events")

is_penalty = df_events['event_type'] == 'penalty'
is_bookkeeping_detail = df_events['detail'].isin(['start', 'end'])
has_player = df_events['player_id'].notna()

# Remove only low-information bookkeeping rows without player association
penalty_bookkeeping_mask = is_penalty & is_bookkeeping_detail & (~has_player)
penalty_removed_count = penalty_bookkeeping_mask.sum()

# Keep player-associated penalty bookkeeping rows for downstream context
kept_bookkeeping_mask = is_penalty & is_bookkeeping_detail & has_player
kept_bookkeeping_count = kept_bookkeeping_mask.sum()

if penalty_removed_count > 0:
    print(f"\nUnassigned penalty bookkeeping events found:")
    print(df_events[penalty_bookkeeping_mask]['detail'].value_counts())

    df_events = df_events[~penalty_bookkeeping_mask].reset_index(drop=True)
    print(f"\n✓ Removed {penalty_removed_count:,} unassigned penalty bookkeeping events (detail='start' or 'end')")
else:
    print("✓ No unassigned penalty bookkeeping events found")

print(f"✓ Retained {kept_bookkeeping_count:,} player-associated penalty bookkeeping rows")
print(f"✓ Events remaining: {len(df_events):,}")

=== 4.1: Remove Unassigned Penalty Bookkeeping Events ===
Starting with 1,799,374 events

Unassigned penalty bookkeeping events found:
detail
start    3850
end      3839
Name: count, dtype: int64

✓ Removed 7,689 unassigned penalty bookkeeping events (detail='start' or 'end')
✓ Retained 0 player-associated penalty bookkeeping rows
✓ Events remaining: 1,791,685


In [8]:
print("\n=== 4.2: Remove Duplicate Goals (Vectorized) ===")
print(f"Starting with {len(df_events):,} events")

# 1. Identify Goal Events
is_goal = df_events['event_type'] == 'goal'

# 2. Identify Consecutive Goals in the Same Sequence
# We shift the dataframe by -1 to compare row[i] with row[i+1]
next_is_goal = is_goal.shift(-1).fillna(False)
same_game = df_events['game_id'] == df_events['game_id'].shift(-1)
same_period = df_events['period'] == df_events['period'].shift(-1)
same_sequence = df_events['sequence_id'] == df_events['sequence_id'].shift(-1)

# A duplicate pair exists where:
# (Current is Goal) AND (Next is Goal) AND (Same Sequence context)
duplicate_pair_start = is_goal & next_is_goal & same_game & same_period & same_sequence

# 3. Determine Which to Drop (Keep the one with player_id)
# Case A: Current has player_id, Next does not -> Drop Next
# We identify these by looking at where the PAIR starts
has_player = df_events['player_id'].notna()
next_no_player = df_events['player_id'].shift(-1).isna()

# Case B: Current has NO player_id, Next does -> Drop Current
current_no_player = df_events['player_id'].isna()
next_has_player = df_events['player_id'].shift(-1).notna()

# Create boolean mask for rows to drop
rows_to_drop = pd.Series(False, index=df_events.index)

# Mark 'Next' for removal if Current keeps
# We find starts where Current=Player, Next=NoPlayer, then shift +1 to mark Next
drop_next_mask = duplicate_pair_start & has_player & next_no_player
rows_to_drop |= drop_next_mask.shift(1).fillna(False)

# Mark 'Current' for removal if Next keeps
drop_current_mask = duplicate_pair_start & current_no_player & next_has_player
rows_to_drop |= drop_current_mask

# 4. Drop and Report
drop_count = rows_to_drop.sum()

if drop_count > 0:
    print(f"\n✓ Found {drop_count:,} duplicate goal rows to remove")
    df_events = df_events[~rows_to_drop].reset_index(drop=True)
    print(f"✓ Removed {drop_count:,} duplicate goal rows")
else:
    print("✓ No duplicate goals found")

print(f"✓ Events remaining: {len(df_events):,}")


=== 4.2: Remove Duplicate Goals (Vectorized) ===
Starting with 1,791,685 events

✓ Found 2,808 duplicate goal rows to remove
✓ Removed 2,808 duplicate goal rows
✓ Events remaining: 1,788,877


In [9]:
print("\n=== 4.3: Remove False-Start Sequences (Faceoff-Only Sequences) - Vectorized ===")
print(f"Starting with {len(df_events):,} events")

# 1. Define Valid & Invalid Types
# We want to identify sequences that contain ONLY {'faceoff', 'whistle'}
# But MUST contain 'faceoff' to be a false start (a sequence of just whistles is weird but different)
valid_false_start_types = {'faceoff', 'whistle'}

# 2. Vectorized Checks per Sequence
# Group by game and sequence
grouped = df_events.groupby(['game_id', 'sequence_id'])['event_type']

# Check A: Does the sequence contain any event NOT in {faceoff, whistle}?
# We use .isin() and then transform('any') to flag sequences with "real" hockey actions
has_real_action = (~df_events['event_type'].isin(valid_false_start_types)).groupby([df_events['game_id'], df_events['sequence_id']]).transform('any')

# Check B: Does the sequence actually contain a faceoff? (To distinguish from empty/weird data)
has_faceoff = (df_events['event_type'] == 'faceoff').groupby([df_events['game_id'], df_events['sequence_id']]).transform('any')

# 3. Identify False Starts
# A False Start Sequence = (No Real Action) AND (Has Faceoff)
is_false_start = (~has_real_action) & has_faceoff

# 4. Report and Remove
false_start_events = df_events[is_false_start]
num_events = len(false_start_events)
num_sequences = false_start_events[['game_id', 'sequence_id']].drop_duplicates().shape[0]

if num_events > 0:
    print(f"\n✓ Found {num_sequences:,} false-start sequences containing {num_events:,} events")
    
    # Optional: Print a few examples
    examples = false_start_events.head(10)
    print("Sample removed rows:")
    print(examples[['game_id', 'sequence_id', 'event_type']].to_string(index=False))
    
    # Filter out
    df_events = df_events[~is_false_start].reset_index(drop=True)
    print(f"✓ Removed {num_events:,} events")
else:
    print("✓ No false-start sequences found")

print(f"✓ Events remaining: {len(df_events):,}")


=== 4.3: Remove False-Start Sequences (Faceoff-Only Sequences) - Vectorized ===
Starting with 1,788,877 events

✓ Found 112 false-start sequences containing 448 events
Sample removed rows:
                             game_id  sequence_id event_type
07c3dfdb-28d2-df56-d0f5-a20f2016aee4           29    faceoff
07c3dfdb-28d2-df56-d0f5-a20f2016aee4           29    faceoff
07c3dfdb-28d2-df56-d0f5-a20f2016aee4           29    faceoff
07c3dfdb-28d2-df56-d0f5-a20f2016aee4           29    whistle
08608329-1e29-482c-e822-3d2587d54705           18    faceoff
08608329-1e29-482c-e822-3d2587d54705           18    faceoff
08608329-1e29-482c-e822-3d2587d54705           18    faceoff
08608329-1e29-482c-e822-3d2587d54705           18    whistle
08608329-1e29-482c-e822-3d2587d54705           22    faceoff
08608329-1e29-482c-e822-3d2587d54705           22    faceoff
✓ Removed 448 events
✓ Events remaining: 1,788,429


In [10]:
print("\n=== 4.3.1: Preview Residual Post-Sequence Actions (Whistle OR Goal) - Fixed ===")
print(f"Starting with {len(df_events):,} events")

# 1. Identify Sequence Terminators (Whistle OR Goal)
# ---------------------------------------------------------
# FIX: Assign as a column so groupby can find it
df_events['is_terminator'] = df_events['event_type'].isin(['whistle', 'goal'])

# 2. Vectorized Scan (Propagate "Ended" State)
# ---------------------------------------------------------
# Group by sequence. 'cummax' propagates True from the first terminator to the end of the group.
# This marks every row AFTER the terminator (and the terminator itself) as True.
has_terminated = df_events.groupby(['game_id', 'period', 'sequence_id'])['is_terminator'].transform('cummax')

# 3. Filter for Residuals
# ---------------------------------------------------------
# We want rows where:
# - The sequence has already "ended" (has_terminated is True)
# - The event is NOT the terminator itself (we want the stuff AFTER the whistle/goal)
# - The event is NOT a faceoff (start of next sequence logic)
excluded_types = {'whistle', 'goal', 'faceoff'}
is_excluded = df_events['event_type'].isin(excluded_types)

# We use the column we just created
is_terminator_col = df_events['is_terminator']

# The logic: "Sequence is over" AND "This isn't the thing that ended it" AND "This isn't a faceoff"
is_residual = has_terminated & (~is_terminator_col) & (~is_excluded)

# 4. Extract Results
# ---------------------------------------------------------
residual_df = df_events[is_residual].copy()

if residual_df.empty:
    print("✓ No post-sequence actions remain before the next sequence's faceoff")
else:
    print(f"Found {len(residual_df):,} residual events.")
    counts = residual_df['event_type'].value_counts()
    print("Events appearing between Terminator (Whistle/Goal) and next Faceoff:")
    print(counts)
    print("\nSample rows:")
    display(residual_df[['game_id', 'period', 'sequence_id', 'event_type', 'description']].head(10))

# Cleanup temporary column
df_events = df_events.drop(columns=['is_terminator'])

print(f"✓ Checked residual actions (Vectorized)")


=== 4.3.1: Preview Residual Post-Sequence Actions (Whistle OR Goal) - Fixed ===
Starting with 1,788,429 events
Found 726 residual events.
Events appearing between Terminator (Whistle/Goal) and next Faceoff:
event_type
penalty         368
penaltydrawn    358
Name: count, dtype: int64

Sample rows:


,game_id,period,sequence_id,event_type,description
728,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,12,penalty,ROUGHING PENALTY OZ
729,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,12,penaltydrawn,ROUGHING PENALTY DRAWN DZ
4588,00f1ee7c-b2e4-3fee-b8ba-37158dc3166d,1,11,penaltydrawn,ROUGHING PENALTY DRAWN OZ
4589,00f1ee7c-b2e4-3fee-b8ba-37158dc3166d,1,11,penalty,ROUGHING PENALTY DZ
17383,02baa060-7e1d-95ae-3186-749790963b95,3,47,penalty,UNSPORTSMANLIKE CONDUCT PENALTY NZ
34243,076e3cb2-02fb-069f-cb82-a45c5a432ed1,1,12,penalty,ROUGHING PENALTY OZ
34244,076e3cb2-02fb-069f-cb82-a45c5a432ed1,1,12,penaltydrawn,ROUGHING PENALTY DRAWN DZ
38173,07c3dfdb-28d2-df56-d0f5-a20f2016aee4,1,20,penaltydrawn,SLASHING PENALTY DRAWN DZ
38174,07c3dfdb-28d2-df56-d0f5-a20f2016aee4,1,20,penalty,SLASHING PENALTY OZ
39945,07c3dfdb-28d2-df56-d0f5-a20f2016aee4,3,56,penalty,ELBOWING PENALTY OZ


✓ Checked residual actions (Vectorized)


In [11]:
print("\n=== 4.4: Remove Post-Sequence Residual Events (Context-Aware) ===")
print(f"Starting with {len(df_events):,} events")

# 1. Identify Event Types & Assign Columns
# ---------------------------------------------------------
df_events['is_goal'] = (df_events['event_type'] == 'goal')
df_events['is_whistle'] = (df_events['event_type'] == 'whistle')
group_cols = ['game_id', 'period', 'sequence_id']

# 2. Identify Rows Following a Goal
# ---------------------------------------------------------
has_goal_occurred = df_events.groupby(group_cols)['is_goal'].transform(
    lambda x: x.cummax().shift(1).fillna(False)
).astype(bool)

# 3. Identify Rows Following a Whistle
# ---------------------------------------------------------
has_whistle_occurred = df_events.groupby(group_cols)['is_whistle'].transform(
    lambda x: x.cummax().shift(1).fillna(False)
).astype(bool)

# 4. Define Removal Logic
# ---------------------------------------------------------
# Rule 1: Remove anything that follows a Goal.
# Rule 2: Remove anything that follows a Whistle, UNLESS:
#         - it is a Goal, OR
#         - it is a player-associated penalty/penaltydrawn event.
is_player_penalty_after_whistle = (
    has_whistle_occurred
    & df_events['event_type'].isin(['penalty', 'penaltydrawn'])
    & df_events['player_id'].notna()
)

rows_to_remove = (
    has_goal_occurred
    | (has_whistle_occurred & (~df_events['is_goal']) & (~is_player_penalty_after_whistle))
)

# 5. Filter and Report
# ---------------------------------------------------------
num_removed = rows_to_remove.sum()
num_preserved_penalties = is_player_penalty_after_whistle.sum()

if num_removed > 0:
    removed_counts = df_events[rows_to_remove]['event_type'].value_counts()
    print(f"\n✓ Found {num_removed:,} post-terminator events to remove")
    print("  Summary by event type:")
    for et, count in removed_counts.items():
        print(f"    - {et}: {count}")

    if num_preserved_penalties > 0:
        print(f"\n✓ Preserving {num_preserved_penalties:,} post-whistle player-associated penalty events")

    df_events = df_events[~rows_to_remove].reset_index(drop=True)
    print(f"✓ Removed {num_removed:,} rows")
else:
    print("\n✓ No post-terminator events found")

# Cleanup temporary columns
df_events = df_events.drop(columns=['is_goal', 'is_whistle'])

print(f"✓ Events remaining: {len(df_events):,}")


=== 4.4: Remove Post-Sequence Residual Events (Context-Aware) ===
Starting with 1,788,429 events

✓ Found 38 post-terminator events to remove
  Summary by event type:
    - penalty: 19
    - penaltydrawn: 17
    - goal: 2

✓ Preserving 690 post-whistle player-associated penalty events
✓ Removed 38 rows
✓ Events remaining: 1,788,391


In [12]:
# Verify whistle transition rules with contextual exception handling
print("\n=== 4.4.1: Confirm Whistle Transition Logic (Context-Aware) ===")
print(f"Starting with {len(df_events):,} events")

is_whistle = df_events['event_type'] == 'whistle'
next_event = df_events.shift(-1)

same_game = next_event['game_id'] == df_events['game_id']
same_period = next_event['period'] == df_events['period']
sequence_change = next_event['sequence_id'] != df_events['sequence_id']
same_sequence = next_event['sequence_id'] == df_events['sequence_id']
faceoff_follow = next_event['event_type'] == 'faceoff'

# Allowed in-sequence continuation: post-whistle player-associated penalties
penalty_context_follow = (
    next_event['event_type'].isin(['penalty', 'penaltydrawn'])
    & next_event['player_id'].notna()
    & same_game
    & same_period
    & same_sequence
)

period_or_game_change = (next_event['game_id'] != df_events['game_id']) | (next_event['period'] != df_events['period'])
end_of_data = next_event['event_type'].isna()

valid_transition = (
    (faceoff_follow & same_game & same_period & sequence_change)
    | penalty_context_follow
    | period_or_game_change
    | end_of_data
)

invalid_whistles = df_events[is_whistle & ~valid_transition]

if invalid_whistles.empty:
    print("✓ All whistles now follow valid transitions (faceoff/new sequence, boundary change, or player-associated penalty context)")
else:
    print(f"⚠ {len(invalid_whistles):,} whistle(s) still followed by unexpected intra-sequence events:")
    display(invalid_whistles[['game_id', 'period', 'sequence_id', 'period_time', 'event_type']].head(20))

print(f"✓ Whistles checked: {is_whistle.sum():,}")
print(f"✓ Valid post-whistle player-penalty continuations: {penalty_context_follow[is_whistle].sum():,}")


=== 4.4.1: Confirm Whistle Transition Logic (Context-Aware) ===
Starting with 1,788,391 events
✓ All whistles now follow valid transitions (faceoff/new sequence, boundary change, or player-associated penalty context)
✓ Whistles checked: 25,254
✓ Valid post-whistle player-penalty continuations: 244


In [13]:
output_path_clean_1 = OUTPUT_DIR / "events_clean_1.parquet"
df_events.to_parquet(output_path_clean_1, index=False)
print(f"✓ Saved df_events to {output_path_clean_1}")

✓ Saved df_events to x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\Processed Sequence Data\events_clean_1.parquet


In [14]:
#Load back and confirm integrity
try:
    df_events = pd.read_parquet(output_path_clean_1)
    if len(df_events) == len(df_events):
        print(f"✓ Successfully loaded back {len(df_events):,} events from {output_path_clean_1}")
    else:
        print(f"⚠ Row count mismatch: saved {len(df_events):,} but loaded {len(df_events):,}")
except Exception as e:
    print(f"⚠ Failed to load back events from {output_path_clean_1}: {e}")

✓ Successfully loaded back 1,788,391 events from x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\Processed Sequence Data\events_clean_1.parquet


In [15]:
print("\n=== 4.5: Scan & Identify Mergeable Metadata Events (Vectorized) ===")

# Define types
# Keep 'reception' as explicit event context; only merge/remove failedpasslocation metadata
mergeable_types = ['failedpasslocation']
kept_context_types = ['reception']
excluded_types = ['receptionprevention', 'block']

# 1. Separate Data for Analysis
# ---------------------------------------------------------
df_passes = df_events[df_events['event_type'] == 'pass'].copy()
df_meta = df_events[df_events['event_type'].isin(mergeable_types)].copy()

# CRITICAL FIX: Create a distinct column for the pass ID so it survives the merge
df_passes['pass_event_id'] = df_passes['sl_event_id']

# 2. Perform Backward Search (Vectorized)
# ---------------------------------------------------------
scan_merge = pd.merge_asof(
    df_meta.sort_values('sl_event_id'),
    df_passes.sort_values('sl_event_id')[['game_id', 'period', 'sequence_id', 'sl_event_id', 'player_id', 'pass_event_id']],
    on='sl_event_id',
    by=['game_id', 'period', 'sequence_id'],
    direction='backward',
    suffixes=('_meta', '_pass')
)

# 3. Calculate Statistics
# ---------------------------------------------------------
print("\n--- Scanning mergeable pass metadata (optimized) ---")

for m_type in mergeable_types:
    subset = scan_merge[scan_merge['event_type'] == m_type]
    total = len(subset)

    if total == 0:
        print(f"\n{m_type}: 0 total")
        continue

    linked = subset[subset['pass_event_id'].notna()]
    orphans = subset[subset['pass_event_id'].isna()]

    print(f"\n{m_type}: {total:,} total")
    print(f"  ✓ Can link to pass: {len(linked):,}")
    print(f"  ✗ Orphaned: {len(orphans):,}")

print("\n--- Kept context event types ---")
for keep_type in kept_context_types:
    count = (df_events['event_type'] == keep_type).sum()
    print(f"{keep_type}: {count:,} retained as standalone events")

print("\n--- Excluded metadata (Independent Actions) ---")
for ex_type in excluded_types:
    count = (df_events['event_type'] == ex_type).sum()
    print(f"{ex_type}: {count:,} total")


=== 4.5: Scan & Identify Mergeable Metadata Events (Vectorized) ===

--- Scanning mergeable pass metadata (optimized) ---

failedpasslocation: 97,366 total
  ✓ Can link to pass: 97,366
  ✗ Orphaned: 0

--- Kept context event types ---
reception: 304,903 retained as standalone events

--- Excluded metadata (Independent Actions) ---
receptionprevention: 2,353 total
block: 66,233 total


In [16]:
print("\n=== 4.6: Merge Pass Outcomes (Vectorized - Context-Aware) ===")
print(f"Starting with {len(df_events):,} events")

# 1. Setup Order Column
# ---------------------------------------------------------
order_col = 'sl_event_id'
created_order_col = False
if order_col not in df_events.columns:
    created_order_col = True
    order_col = '_temp_sl_event_id'
    df_events[order_col] = range(len(df_events))

# 2. Separate Data & Store Original Indices
# ---------------------------------------------------------
# Keep 'reception' rows as contextual events in continuous-state modeling
metadata_types = ['failedpasslocation']

# Track which rows are metadata (to remove later)
is_metadata = df_events['event_type'].isin(metadata_types)
metadata_indices_to_remove = df_events[is_metadata].index.tolist()
print(f"Metadata rows marked for removal (failedpasslocation only): {len(metadata_indices_to_remove):,}")
print(f"Reception rows retained: {(df_events['event_type'] == 'reception').sum():,}")

# Extract passes with their df_events index
df_passes = df_events[df_events['event_type'] == 'pass'].copy()
df_passes['pass_df_index'] = df_passes.index

# Extract outcomes
df_outcomes = df_events[is_metadata].copy()

# 3. Prepare Columns for Merge
# ---------------------------------------------------------
df_outcomes = df_outcomes.rename(columns={
    'x_adj': 'dest_x_adj',
    'y_adj': 'dest_y_adj',
    'description': 'outcome_desc'
})

# 4. Perform Backward Merge (Outcome -> Preceding Pass)
# ---------------------------------------------------------
merged = pd.merge_asof(
    df_outcomes.sort_values(order_col),
    df_passes.sort_values(order_col)[[order_col, 'game_id', 'period', 'sequence_id', 'pass_df_index']],
    on=order_col,
    by=['game_id', 'period', 'sequence_id'],
    direction='backward',
    suffixes=('', '_pass')
)

matched = merged[merged['pass_df_index'].notna()].copy()
print(f"✓ Successfully linked {len(matched):,} outcome events to passes")

# 5. Update df_events (Passes)
# ---------------------------------------------------------
if not matched.empty:
    grouped = matched.groupby('pass_df_index').last()

    update_indices = grouped.index.astype(int)

    # Update coordinates
    df_events.loc[update_indices, 'dest_x_adj'] = grouped['dest_x_adj'].values
    df_events.loc[update_indices, 'dest_y_adj'] = grouped['dest_y_adj'].values

    # Update descriptions
    orig_desc = df_events.loc[update_indices, 'description'].fillna('')
    new_desc = grouped['outcome_desc'].fillna('').values
    combined = (orig_desc + ' | ' + new_desc).str.strip(' |')
    df_events.loc[update_indices, 'description'] = combined

    print(f"✓ Updated {len(update_indices):,} pass events with failedpasslocation metadata")

# 6. Remove Merged Metadata Rows
# ---------------------------------------------------------
df_events = df_events.drop(index=metadata_indices_to_remove).reset_index(drop=True)
print(f"✓ Removed {len(metadata_indices_to_remove):,} metadata rows (failedpasslocation only)")
print(f"✓ Retained reception rows for downstream state context")

# Cleanup temp columns
if created_order_col:
    df_events = df_events.drop(columns=[order_col], errors='ignore')

print(f"\n✓ Events remaining: {len(df_events):,}")


=== 4.6: Merge Pass Outcomes (Vectorized - Context-Aware) ===
Starting with 1,788,391 events
Metadata rows marked for removal (failedpasslocation only): 97,366
Reception rows retained: 304,903
✓ Successfully linked 97,366 outcome events to passes
✓ Updated 97,357 pass events with failedpasslocation metadata
✓ Removed 97,366 metadata rows (failedpasslocation only)
✓ Retained reception rows for downstream state context

✓ Events remaining: 1,691,025


In [17]:
# Validate merge results and data quality
print("\n=== 4.7: Validate Merge & Data Quality ===")
print(f"Total events after merges: {len(df_events):,}")

# Check 1: failedpasslocation removed; reception intentionally retained
print("\n--- Check 1: Pass Metadata Handling ---")
failedpass_remaining = (df_events['event_type'] == 'failedpasslocation').sum()
reception_remaining = (df_events['event_type'] == 'reception').sum()

if failedpass_remaining == 0:
    print("✓ failedpasslocation: 0 remaining (all merged)")
else:
    print(f"⚠ failedpasslocation: {failedpass_remaining} remaining")

print(f"✓ reception: {reception_remaining} retained as standalone context events")

# Check 2: Defensive actions preserved
print("\n--- Check 2: Defensive Actions Preserved ---")
defensive_actions = ['receptionprevention', 'block']
for action_type in defensive_actions:
    count = (df_events['event_type'] == action_type).sum()
    print(f"✓ {action_type}: {count} total (independent events, not merged)")

# Check 3: Dest coordinates are properly populated
print("\n--- Check 3: Destination Coordinates ---")
if 'dest_x_adj' in df_events.columns:
    dest_x_populated = df_events['dest_x_adj'].notna().sum()
    print(f"✓ dest_x_adj: {dest_x_populated} values populated")
else:
    print("⚠ dest_x_adj column not found (may not have been created)")

if 'dest_y_adj' in df_events.columns:
    dest_y_populated = df_events['dest_y_adj'].notna().sum()
    print(f"✓ dest_y_adj: {dest_y_populated} values populated")
else:
    print("⚠ dest_y_adj column not found (may not have been created)")

# Check 4: Unassigned penalty bookkeeping removed; player-associated kept
print("\n--- Check 4: Penalty Bookkeeping Policy ---")
unassigned_penalty_bookkeeping = df_events[
    (df_events['event_type'] == 'penalty')
    & (df_events['detail'].isin(['start', 'end']))
    & (df_events['player_id'].isna())
]
player_penalty_bookkeeping = df_events[
    (df_events['event_type'] == 'penalty')
    & (df_events['detail'].isin(['start', 'end']))
    & (df_events['player_id'].notna())
]

if len(unassigned_penalty_bookkeeping) == 0:
    print("✓ No unassigned penalty bookkeeping events remaining")
else:
    print(f"⚠ {len(unassigned_penalty_bookkeeping)} unassigned penalty bookkeeping events still present")

print(f"✓ Player-associated penalty bookkeeping retained: {len(player_penalty_bookkeeping)}")

# Check 5: No duplicate goals (Vectorized)
print("\n--- Check 5: Duplicate Goals (Vectorized) ---")
is_goal = df_events['event_type'] == 'goal'
next_is_goal = is_goal.shift(-1).fillna(False)
same_sequence = df_events['sequence_id'] == df_events['sequence_id'].shift(-1)
same_game = df_events['game_id'] == df_events['game_id'].shift(-1)

duplicate_goal_mask = is_goal & next_is_goal & same_sequence & same_game
duplicate_goals = duplicate_goal_mask.sum()

if duplicate_goals == 0:
    print("✓ No duplicate goal patterns found")
else:
    print(f"⚠ {duplicate_goals} potential duplicate goal patterns detected")

# Check 6: No false-start faceoff patterns (Vectorized)
print("\n--- Check 6: False-Start Faceoffs (Vectorized) ---")
is_whistle = df_events['event_type'] == 'whistle'
is_faceoff = df_events['event_type'] == 'faceoff'

prev_is_whistle = is_whistle.shift(1).fillna(False)
next_is_whistle = is_whistle.shift(-1).fillna(False)

same_seq_prev = df_events['sequence_id'] == df_events['sequence_id'].shift(1)

false_start_mask = prev_is_whistle & is_faceoff & next_is_whistle & same_seq_prev
false_starts = false_start_mask.sum()

if false_starts == 0:
    print("✓ No false-start faceoff patterns found")
else:
    print(f"⚠ {false_starts} false-start faceoff patterns detected")

# Check 7: Event type distribution
print("\n--- Check 7: Event Type Distribution (Top 15) ---")
top_event_types = df_events['event_type'].value_counts().head(15)
for event_type, count in top_event_types.items():
    pct = (count / len(df_events)) * 100
    print(f"  {event_type}: {count:,} ({pct:.1f}%)")

print(f"\n{'='*70}")
print("✓ Data cleaning pre-sequence-order stages complete")
print(f"{'='*70}")

# Save clean events
output_path_clean_2 = OUTPUT_DIR / "events_clean_2.parquet"
df_events.to_parquet(output_path_clean_2, index=False)
print(f"✓ Saved final cleaned events to {output_path_clean_2}")


=== 4.7: Validate Merge & Data Quality ===
Total events after merges: 1,691,025

--- Check 1: Pass Metadata Handling ---
✓ failedpasslocation: 0 remaining (all merged)
✓ reception: 304903 retained as standalone context events

--- Check 2: Defensive Actions Preserved ---
✓ receptionprevention: 2353 total (independent events, not merged)
✓ block: 66233 total (independent events, not merged)

--- Check 3: Destination Coordinates ---
✓ dest_x_adj: 97357 values populated
✓ dest_y_adj: 97357 values populated

--- Check 4: Penalty Bookkeeping Policy ---
✓ No unassigned penalty bookkeeping events remaining
✓ Player-associated penalty bookkeeping retained: 0

--- Check 5: Duplicate Goals (Vectorized) ---
✓ No duplicate goal patterns found

--- Check 6: False-Start Faceoffs (Vectorized) ---
✓ No false-start faceoff patterns found

--- Check 7: Event Type Distribution (Top 15) ---
  pass: 402,670 (23.8%)
  lpr: 338,836 (20.0%)
  reception: 304,903 (18.0%)
  carry: 110,362 (6.5%)
  faceoff: 84,2

## 5. Validate and Enforce Sequence Order

In [24]:
# Load back and confirm integrity
try:
    df_events = pd.read_parquet(output_path_clean_2)
    if len(df_events) == len(df_events):
        print(f"✓ Successfully loaded back {len(df_events):,} events from {output_path_clean_2}")
    else:
        print(f"⚠ Row count mismatch: saved {len(df_events):,} but loaded {len(df_events):,}")
except Exception as e:
    print(f"⚠ Failed to load back events from {output_path_clean_2}: {e}") 
    

✓ Successfully loaded back 1,691,025 events from x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\Processed Sequence Data\events_clean_2.parquet


In [25]:
# Check the count of sequence ending events
last_events = df_events.groupby(['game_id', 'sequence_id']).tail(1)
eos_distribution = last_events['event_type'].value_counts()
print("\nSequence Ending Event Distribution:")
print(eos_distribution)
print(f"\nTotal sequences: {len(last_events):,}")
print(f"Whistle + Goal coverage: {(eos_distribution.get('whistle', 0) + eos_distribution.get('goal', 0)) / len(last_events) * 100:.2f}%")


Sequence Ending Event Distribution:
event_type
whistle                   25010
goal                       2810
penaltydrawn                144
penalty                     100
shot                         10
lpr                           4
pass                          4
puckprotection                3
reception                     3
block                         2
dumpout                       2
controlledentryagainst        2
dumpin                        1
Name: count, dtype: int64

Total sequences: 28,095
Whistle + Goal coverage: 99.02%


In [26]:
# Comprehensive sequence ending cleanup and validation
print("="*70)
print("=== 5. Final Sequence Order & EOS Token Cleanup ===")
print("="*70)
print(f"\nStarting with {len(df_events):,} events")

# Boundary tolerance for period-end detection (seconds)
PERIOD_END_SEC = 1200.0
BOUNDARY_TOL_SEC = 20.0

# Step 1: Natural sort
# ---------------------------------------------------------
print("\n--- Step 1: Sort events by natural order ---")
sort_cols = ['game_id', 'period', 'sequence_id', 'period_time']
if 'sl_event_id' in df_events.columns:
    sort_cols.append('sl_event_id')

df_events = df_events.sort_values(by=sort_cols, ascending=True).reset_index(drop=True)
print("✓ Sorted by: game -> period -> sequence -> time -> event_id")

# Step 1.5: Normalize existing end_of_period events
# ---------------------------------------------------------
print("\n--- Step 1.5: Normalize existing end_of_period events ---")
period_time_num = pd.to_numeric(df_events['period_time'], errors='coerce')
near_period_boundary = (
    (period_time_num >= (PERIOD_END_SEC - BOUNDARY_TOL_SEC))
    | (period_time_num <= BOUNDARY_TOL_SEC)
)

existing_eop_mask = df_events['event_type'] == 'end_of_period'
invalid_eop_mask = existing_eop_mask & (~near_period_boundary)

if invalid_eop_mask.any():
    invalid_count = int(invalid_eop_mask.sum())
    df_events.loc[invalid_eop_mask, 'event_type'] = 'whistle'
    if 'detail' in df_events.columns:
        df_events.loc[invalid_eop_mask, 'detail'] = 'EOS_whistle_normalized'
    if 'description' in df_events.columns:
        df_events.loc[invalid_eop_mask, 'description'] = 'Normalized to whistle (mid-period end_of_period)'
    print(f"✓ Normalized {invalid_count:,} mid-period end_of_period events to whistle")
else:
    print("✓ No mid-period end_of_period normalization needed")

# Step 2: Add EOS markers for sequences not ending with whistle/goal/end_of_period
# ---------------------------------------------------------
print("\n--- Step 2: Add EOS markers for non-standard endings ---")
print("Policy: insert 'whistle' by default; use 'end_of_period' only near period boundary")

# Find sequences that don't end with whistle, goal, or valid end_of_period
last_events = df_events.groupby(['game_id', 'sequence_id']).tail(1)
non_standard = last_events[~last_events['event_type'].isin(['whistle', 'goal', 'end_of_period'])].copy()

print(f"Found {len(non_standard)} sequences not ending with whistle/goal/end_of_period")

if len(non_standard) > 0:
    print(f"\nSequence Ending Event Distribution:")
    print(last_events['event_type'].value_counts())

    eos_rows = []
    for _, event in non_standard.iterrows():
        eos = event.copy()

        period_time_val = pd.to_numeric(pd.Series([event['period_time']]), errors='coerce').iloc[0]
        near_boundary = (
            pd.notna(period_time_val)
            and (
                (period_time_val >= (PERIOD_END_SEC - BOUNDARY_TOL_SEC))
                or (period_time_val <= BOUNDARY_TOL_SEC)
            )
        )

        if near_boundary:
            eos['event_type'] = 'end_of_period'
            eos['detail'] = 'EOS_end_of_period_marker'
            eos['description'] = f"End of period EOS (after {event['event_type']})"
        else:
            eos['event_type'] = 'whistle'
            eos['detail'] = 'EOS_whistle_inserted'
            eos['description'] = f"Inserted whistle EOS (after {event['event_type']})"

        eos_rows.append(eos)

    eos_df = pd.DataFrame(eos_rows)
    df_events = pd.concat([df_events, eos_df], ignore_index=True)
    df_events = df_events.sort_values(by=sort_cols, ascending=True).reset_index(drop=True)

    added_counts = eos_df['event_type'].value_counts()
    print(f"✓ Added {len(eos_df)} EOS rows")
    print(f"  - whistle inserted: {int(added_counts.get('whistle', 0))}")
    print(f"  - end_of_period inserted: {int(added_counts.get('end_of_period', 0))}")
else:
    print("✓ All sequences already end with whistle, goal, or valid end_of_period")

# Step 3: Validate final EOS token distribution
# ---------------------------------------------------------
print("\n--- Step 3: Validate EOS token distribution ---")

last_events_final = df_events.groupby(['game_id', 'sequence_id']).tail(1)
eos_types = last_events_final['event_type'].value_counts()

print(f"\nLast event types across {len(last_events_final):,} sequences:")
for event_type, count in eos_types.items():
    pct = (count / len(last_events_final)) * 100
    print(f"  {event_type}: {count:,} ({pct:.1f}%)")

# Calculate coverage
whistle_goal_pct = ((eos_types.get('whistle', 0) + eos_types.get('goal', 0)) / len(last_events_final)) * 100
print(f"\n✓ Whistle + Goal coverage: {whistle_goal_pct:.1f}% (primary EOS tokens)")

if eos_types.get('end_of_period', 0) > 0:
    eop_pct = (eos_types.get('end_of_period', 0) / len(last_events_final)) * 100
    print(f"✓ End-of-period markers: {eop_pct:.1f}%")

print(f"\n✓ Total events: {len(df_events):,}")
print(f"✓ Total sequences: {len(last_events_final):,}")

# Step 4: Save cleaned data to checkpoint
# ---------------------------------------------------------
print("\n--- Step 4: Save checkpoint ---")
output_path_clean_3 = OUTPUT_DIR / "events_clean_3.parquet"
df_events.to_parquet(output_path_clean_3, index=False)
print(f"✓ Saved to: {output_path_clean_3}")

print("\n" + "="*70)
print("✓ SECTION 5 COMPLETE - Sequence order validated and saved")
print("="*70)

=== 5. Final Sequence Order & EOS Token Cleanup ===

Starting with 1,691,025 events

--- Step 1: Sort events by natural order ---
✓ Sorted by: game -> period -> sequence -> time -> event_id

--- Step 1.5: Normalize existing end_of_period events ---
✓ No mid-period end_of_period normalization needed

--- Step 2: Add EOS markers for non-standard endings ---
Policy: insert 'whistle' by default; use 'end_of_period' only near period boundary
Found 275 sequences not ending with whistle/goal/end_of_period

Sequence Ending Event Distribution:
event_type
whistle                   25010
goal                       2810
penaltydrawn                144
penalty                     100
shot                         10
lpr                           4
pass                          4
puckprotection                3
reception                     3
block                         2
dumpout                       2
controlledentryagainst        2
dumpin                        1
Name: count, dtype: int64
✓ Adde

In [27]:
# Diagnostic: Validate EOS token assignment
print("\n=== 5.1: Diagnostic - EOS Token Validation ===")

# Check 1: Verify all sequences end with proper EOS tokens
last_events_final = df_events.groupby(['game_id', 'sequence_id']).tail(1)
eos_distribution = last_events_final['event_type'].value_counts()

print(f"\n--- Final EOS Token Distribution ---")
print(f"Total sequences: {len(last_events_final):,}\n")
for event_type, count in eos_distribution.items():
    pct = (count / len(last_events_final)) * 100
    print(f"  {event_type:20s}: {count:5,} ({pct:5.2f}%)")

# Check 2: Inspect end_of_period markers
print(f"\n--- Check: end_of_period Markers ---")
eop_markers = df_events[df_events['event_type'] == 'end_of_period']
if len(eop_markers) > 0:
    print(f"Total end_of_period markers: {len(eop_markers)}")
    print(f"\nSample sequences with end_of_period (showing 3 examples):")
    
    for idx, (game_id, seq_id) in enumerate(eop_markers[['game_id', 'sequence_id']].head(3).values):
        seq_events = df_events[
            (df_events['game_id'] == game_id) & 
            (df_events['sequence_id'] == seq_id)
        ].tail(6)  # Show last 6 events
        
        print(f"\n  Example {idx+1}: Game {game_id[:8]}..., Seq {seq_id}")
        print(f"  Last 6 events:")
        for _, evt in seq_events.iterrows():
            marker = " <- end_of_period EOS" if evt['event_type'] == 'end_of_period' else ""
            print(f"    {evt['event_type']:20s} | time: {evt['period_time']:6.1f}s{marker}")
else:
    print("✓ No end_of_period markers (all sequences end naturally with whistle/goal)")

# Check 3: Summary statistics
whistle_count = eos_distribution.get('whistle', 0)
goal_count = eos_distribution.get('goal', 0)
eop_count = eos_distribution.get('end_of_period', 0)
other_count = len(last_events_final) - whistle_count - goal_count - eop_count

print(f"\n--- Summary ---")
print(f"✓ Primary EOS tokens (whistle + goal): {whistle_count + goal_count:,} ({((whistle_count + goal_count) / len(last_events_final) * 100):.2f}%)")
print(f"✓ End-of-period markers: {eop_count:,} ({(eop_count / len(last_events_final) * 100):.2f}%)")
if other_count > 0:
    print(f"⚠ Other endings: {other_count:,} ({(other_count / len(last_events_final) * 100):.2f}%)")
else:
    print(f"✓ No unexpected sequence endings")

print(f"\n✓ Validation complete")


=== 5.1: Diagnostic - EOS Token Validation ===

--- Final EOS Token Distribution ---
Total sequences: 28,095

  whistle             : 25,273 (89.96%)
  goal                : 2,810 (10.00%)
  end_of_period       :    12 ( 0.04%)

--- Check: end_of_period Markers ---
Total end_of_period markers: 12

Sample sequences with end_of_period (showing 3 examples):

  Example 1: Game 0c71a655..., Seq 15
  Last 6 events:
    pass                 | time: 1198.7s
    reception            | time: 1199.0s
    whistle              | time: 1200.0s
    penalty              | time: 1200.0s
    penaltydrawn         | time: 1200.0s
    end_of_period        | time: 1200.0s <- end_of_period EOS

  Example 2: Game 13d05644..., Seq 42
  Last 6 events:
    pass                 | time: 1192.3s
    reception            | time: 1193.3s
    whistle              | time: 1200.0s
    penaltydrawn         | time: 1200.0s
    penalty              | time: 1200.0s
    end_of_period        | time: 1200.0s <- end_of_period 

## 6. Clean, Inspect, and Reorder Goal Sequences

This section reloads the latest cleaned checkpoint, removes assist leakage, inspects what happens immediately before goals, and reorders non-standard goal neighbors so sequences more consistently reflect shots preceding goals.

### 6.0 Reload cleaned checkpoint for Section 6 work

Reload the most recent cleaned table (`events_clean_3.parquet`) before applying the inspection and reorder steps.

In [28]:
# Load clean data (this will be the base for all Section 6 transformations)
output_path_clean_3 = OUTPUT_DIR / "events_clean_3.parquet"
print(f"Loading events from {output_path_clean_3.name}...")
df_events = pd.read_parquet(output_path_clean_3)
print(f"✓ Loaded {len(df_events):,} events")
print(f"✓ Total goals in data: {(df_events['event_type'] == 'goal').sum():,}")

Loading events from events_clean_3.parquet...
✓ Loaded 1,691,300 events
✓ Total goals in data: 2,810


### 6.1 Remove assist leakage

Drop any assist rows to avoid label leakage before analyzing sequences.

In [29]:
print("\n=== 6.1: Remove Assists (Data Leakage) ===")
print(f"Starting with {len(df_events):,} events")

assist_mask = df_events['event_type'] == 'assist'
assists_removed = assist_mask.sum()
df_events = df_events[~assist_mask].reset_index(drop=True)

print(f"✓ Removed {assists_removed:,} assist rows")
print(f"✓ Events remaining: {len(df_events):,}")


=== 6.1: Remove Assists (Data Leakage) ===
Starting with 1,691,300 events
✓ Removed 4,793 assist rows
✓ Events remaining: 1,686,507


### 6.2 Inspect events immediately before goals

Analyze the event types that directly precede goals to determine whether additional reordering is needed.

In [30]:
print("\n=== 6.2: Evaluate Events Prior to Goals ===")
print("Analyzing what happens directly before goal events\n")

# Find all goal events
is_goal = df_events['event_type'] == 'goal'
total_goals = is_goal.sum()
print(f"Found {total_goals:,} goal events\n")

# Identify preceding event type for each goal (temporarily, for analysis)
preceding_event_type = df_events.groupby(['game_id', 'sequence_id'])['event_type'].shift(1)

# Analyze BEFORE reordering distribution
print(f"BEFORE REORDERING - Distribution of events immediately before goals:")
preceding_dist = preceding_event_type[is_goal].value_counts()
for event_type, count in preceding_dist.items():
    pct = (count / total_goals) * 100
    print(f"  {event_type:20s}: {count:5,} ({pct:5.1f}%)")

# Identify problematic cases (not shot, deflection, or block)
needs_reorder_mask = is_goal & ~preceding_event_type.isin(['shot', 'deflection', 'block'])
events_to_move = needs_reorder_mask.sum()
print(f"\nEvents to reorder (prior to goal, excluding shot/deflection/block): {events_to_move:,}")

if events_to_move > 0:
    problem_types = preceding_event_type[needs_reorder_mask].value_counts()
    print(f"Types needing reorder:")
    for event_type, count in problem_types.items():
        print(f"  {event_type}: {count}")


=== 6.2: Evaluate Events Prior to Goals ===
Analyzing what happens directly before goal events

Found 2,810 goal events

BEFORE REORDERING - Distribution of events immediately before goals:
  shot                : 2,363 ( 84.1%)
  deflection          :   259 (  9.2%)
  block               :   184 (  6.5%)
  lpr                 :     3 (  0.1%)
  penalty             :     1 (  0.0%)

Events to reorder (prior to goal, excluding shot/deflection/block): 4
Types needing reorder:
  lpr: 3
  penalty: 1


### 6.3 Reorder LPR/penalty events and resync with shot chains

When the event directly before a goal is a penalty or LPR, push it back behind the preceding shot/block chain to maintain a consistent causal ordering.

In [32]:
import gc
import pandas as pd
import numpy as np

print("\n=== 6.3 & 6.4: Reorder & Sync (Strict Goal-Adjacency Only) ===")
print("Strategy: Check the ONE event directly before the goal. If it is LPR/Penalty, move it behind the preceding Shot sequence.\n")

# 1. Setup Sort Order (In-Place)
# ---------------------------------------------------------
df_events.sort_values(['game_id', 'period', 'sequence_id', 'period_time'], inplace=True)
df_events.reset_index(drop=True, inplace=True)
df_events['_sort_key'] = df_events.index.astype(float)

# 2. Define Definitions
# ---------------------------------------------------------
# Movers: The event types we check for at Goal-1
type_mover = {'lpr', 'penalty', 'penaltydrawn'}
# Anchors: The "Shot Sequence" events we want to jump over
type_anchor = {'shot', 'deflection', 'block'}

# 3. Iterate Goal Indices Directly
# ---------------------------------------------------------
goal_indices = df_events.index[df_events['event_type'] == 'goal'].values
print(f"Scanning {len(goal_indices):,} goal sequences...")

count_moved = 0
moved_sequences = [] # List to store IDs of modified sequences for verification

for goal_idx in goal_indices:
    # -----------------------------------------------------
    # CRITICAL CHECK: Strict Adjacency
    # -----------------------------------------------------
    # We only care if the event DIRECTLY preceding the goal is a Mover.
    prev_idx = goal_idx - 1
    if prev_idx < 0: continue
    
    # 1. Check Sequence Integrity
    if df_events.at[prev_idx, 'sequence_id'] != df_events.at[goal_idx, 'sequence_id']:
        continue
        
    # 2. Check Event Type (The "Gatekeeper")
    # If the event before the goal is NOT LPR/Penalty, we stop.
    if df_events.at[prev_idx, 'event_type'] not in type_mover:
        continue

    # -----------------------------------------------------
    # If we pass, we have: [ ... ? ... ] -> [LPR/Penalty] -> [Goal]
    # Now we scan backwards from prev_idx to find the Shot/Block chain
    # -----------------------------------------------------
    
    mover_idx = prev_idx
    anchor_idx = None
    
    # Scan back up to 5 steps from the Mover
    # Look for the START of the shot chain
    start_search = max(0, mover_idx - 5)
    
    for curr_idx in range(mover_idx - 1, start_search - 1, -1):
        # Integrity Check
        if df_events.at[curr_idx, 'sequence_id'] != df_events.at[goal_idx, 'sequence_id']:
            break
            
        etype = df_events.at[curr_idx, 'event_type']
        
        if etype in type_anchor:
            # Found a Shot/Deflection/Block. 
            # This is our new target. Keep scanning back to find the EARLIEST one.
            anchor_idx = curr_idx
        elif etype in type_mover:
            # Another LPR/Penalty? Ignore/Skip, we only move the one touching the goal.
            pass 
        else:
            # Neutral event (Pass, Carry). Chain broken.
            break
            
    # -----------------------------------------------------
    # Apply Move
    # -----------------------------------------------------
    if anchor_idx is not None:
        # We found an Anchor (Shot/Block) preceding the LPR/Penalty
        # Move Mover (currently at Goal-1) to BEFORE Anchor
        
        # Target: Anchor Index - 0.5
        new_key = float(anchor_idx) - 0.5
        target_time = df_events.at[anchor_idx, 'period_time']
        
        df_events.at[mover_idx, '_sort_key'] = new_key
        df_events.at[mover_idx, 'period_time'] = target_time
        
        # Log this sequence for verification
        moved_sequences.append((df_events.at[goal_idx, 'game_id'], df_events.at[goal_idx, 'sequence_id']))
        count_moved += 1

# 4. Final Sort & Cleanup
# ---------------------------------------------------------
if count_moved > 0:
    print(f"✓ Moved & Synced {count_moved:,} events.")
    df_events = df_events.sort_values(['game_id', 'period', 'sequence_id', '_sort_key']).reset_index(drop=True)
else:
    print("✓ No events required reordering.")

df_events = df_events.drop(columns=['_sort_key'])
gc.collect()

print(f"✓ Process Complete. Total events: {len(df_events):,}")

# 5. Targeted Verification
# ---------------------------------------------------------
print("\n=== Verification: Showing ONLY the Modified Sequences ===")

if moved_sequences:
    for i, (g_id, s_id) in enumerate(moved_sequences):
        print(f"\n--- Example {i+1}: Game {g_id} | Seq {s_id} ---")
        
        # Retrieve the sequence from the NEWLY sorted dataframe
        # We find the indices for this game and sequence
        seq_mask = (df_events['game_id'] == g_id) & (df_events['sequence_id'] == s_id)
        
        # Display the relevant rows (Goal and preceding context)
        # We show the whole sequence or just the last few rows
        seq_df = df_events[seq_mask]
        
        if not seq_df.empty:
            # Find goal index relative to this slice
            goal_sub_idx = seq_df[seq_df['event_type'] == 'goal'].index
            if not goal_sub_idx.empty:
                end_pos = goal_sub_idx[-1]
                # Show up to 6 events ending at the goal
                start_pos = max(seq_df.index[0], end_pos - 6)
                display(df_events.loc[start_pos : end_pos][['period_time', 'event_type', 'description']])
            else:
                display(seq_df[['period_time', 'event_type', 'description']])
else:
    print("No events were moved, so nothing to display.")


=== 6.3 & 6.4: Reorder & Sync (Strict Goal-Adjacency Only) ===
Strategy: Check the ONE event directly before the goal. If it is LPR/Penalty, move it behind the preceding Shot sequence.

Scanning 2,810 goal sequences...
✓ Moved & Synced 1 events.
✓ Process Complete. Total events: 1,686,507

=== Verification: Showing ONLY the Modified Sequences ===

--- Example 1: Game fd096949-c719-78a0-d106-0059167fd0c7 | Seq 62 ---


,period_time,event_type,description
1662942,1146.170000000,carry,CONTROLLED EXIT FROM DZ
1662943,1147.570000000,pressure,SHOT PRESSURE
1662944,1147.570000000,carry,CARRY OVER REDLINE
1662945,1147.600000000,penalty,SLASHING PENALTY NZ
1662946,1147.600000000,penaltydrawn,SLASHING PENALTY DRAWN NZ
1662947,1147.600000000,shot,OUTSIDE SHOT FOR ONNET
1662948,1148.970000000,goal,GOAL


### 6.4 Penalty-context diagnostics

Inspect penalty sequences that were adjacent to goals to verify the reordering kept the desired context intact.

In [33]:
print("\n=== Verification: Penalty Context Around Goals ===")

# 1. Identify Goals
goal_indices = df_events.index[df_events['event_type'] == 'goal']

# 2. Scan for Penalties preceding Goals
penalty_examples = []
penalty_types = ['penalty', 'penaltydrawn']

for goal_idx in goal_indices:
    # Look at the 4 events before the goal
    start_idx = max(0, goal_idx - 4)
    # Get the window including the goal
    window = df_events.loc[start_idx : goal_idx].copy()
    
    # Ensure we are in the same sequence/game (handle boundary issues)
    goal_row = df_events.loc[goal_idx]
    window = window[
        (window['game_id'] == goal_row['game_id']) & 
        (window['sequence_id'] == goal_row['sequence_id'])
    ]
    
    # Check if a penalty exists in this window (excluding the goal itself)
    preceding_events = window.iloc[:-1] # Drop the goal from check
    if preceding_events['event_type'].isin(penalty_types).any():
        penalty_examples.append((goal_row['game_id'], goal_row['sequence_id']))

# 3. Display Results
if not penalty_examples:
    print("No sequences found where a Penalty immediately precedes a Goal (within 4 events).")
else:
    print(f"Found {len(penalty_examples)} sequences with a Penalty near a Goal.")
    
    # Show top 5 unique examples
    unique_examples = sorted(list(set(penalty_examples)))[:5]
    
    for game_id, seq_id in unique_examples:
        print(f"\n--- Sequence with Penalty (Game {game_id}, Seq {seq_id}) ---")
        
        # Get the full sequence slice
        seq_df = df_events[(df_events['game_id'] == game_id) & (df_events['sequence_id'] == seq_id)]
        
        # Find goal in this sequence
        seq_goal_idx = seq_df[seq_df['event_type'] == 'goal'].index[-1]
        
        # Show context: 5 events before goal + goal
        start_display = max(seq_df.index[0], seq_goal_idx - 5)
        display_slice = df_events.loc[start_display : seq_goal_idx]
        
        display(display_slice[['period_time', 'event_type', 'description', 'player_id']])


=== Verification: Penalty Context Around Goals ===
Found 4 sequences with a Penalty near a Goal.

--- Sequence with Penalty (Game a80c29d0-62cc-ebaf-fdc8-acfe8f6c10da, Seq 42) ---


,period_time,event_type,description,player_id
1033579,356.330000000,penaltydrawn,SLASHING PENALTY DRAWN OZ,6e6168a5-df29-f732-cbd8-5e5d3ccfc780
1033580,356.330000000,penalty,SLASHING PENALTY DZ,251a2f59-76d1-290c-b7a1-729a090a8aaf
1033581,356.930000000,pressure,SHOT PRESSURE,251a2f59-76d1-290c-b7a1-729a090a8aaf
1033582,356.930000000,lpr,REB LPR OZ+,6e6168a5-df29-f732-cbd8-5e5d3ccfc780
1033583,356.970000000,shot,SLOT SHOT FOR ONNET,6e6168a5-df29-f732-cbd8-5e5d3ccfc780
1033584,357.800000000,goal,GOAL,6e6168a5-df29-f732-cbd8-5e5d3ccfc780



--- Sequence with Penalty (Game b1a83f30-98cb-2f26-763e-69a98220d3e4, Seq 50) ---


,period_time,event_type,description,player_id
1090201,1001.000000000,controlledentryagainst,1on2 CONTROLLED ENTRY AGAINST-,066364b9-b235-6207-1061-3d0155cb2b66
1090202,1001.330000000,penalty,OTHER - UNDISCIPLINED PENALTY NZ,c5c3387a-6e3e-1ae9-824c-315974f6414e
1090203,1001.330000000,penaltydrawn,OTHER - UNDISCIPLINED PENALTY DRAWN NZ,2622a685-e219-c5e7-e7f1-d413e86d75cc
1090204,1001.770000000,pressure,SHOT PRESSURE,066364b9-b235-6207-1061-3d0155cb2b66
1090205,1001.800000000,shot,OUTSIDE SHOT FOR ONNET,8e21faad-e800-5bc9-c0ab-dd31c702e09f
1090206,1002.430000000,goal,GOAL,8e21faad-e800-5bc9-c0ab-dd31c702e09f



--- Sequence with Penalty (Game ddd87f9c-5e66-f63f-3b9a-c74314fb73b9, Seq 42) ---


,period_time,event_type,description,player_id
1444008,53.670000000,reception,O-ZONE PASS RECEPTION,e88acb6b-d5da-6c45-42b6-b77998c3203f
1444009,54.230000000,penaltydrawn,SLASHING PENALTY DRAWN OZ,e88acb6b-d5da-6c45-42b6-b77998c3203f
1444010,54.230000000,penalty,SLASHING PENALTY DZ,a0c15a5d-c635-d29e-06a4-421921529350
1444011,54.270000000,pressure,SHOT PRESSURE,a0c15a5d-c635-d29e-06a4-421921529350
1444012,54.300000000,shot,SLOT SHOT FOR ONNET,e88acb6b-d5da-6c45-42b6-b77998c3203f
1444013,54.700000000,goal,GOAL,e88acb6b-d5da-6c45-42b6-b77998c3203f



--- Sequence with Penalty (Game fd096949-c719-78a0-d106-0059167fd0c7, Seq 62) ---


,period_time,event_type,description,player_id
1662943,1147.570000000,pressure,SHOT PRESSURE,cce167f2-c8fd-27c9-a350-0a4931c5b132
1662944,1147.570000000,carry,CARRY OVER REDLINE,3cfc8139-e81a-8949-e3fc-0a6728b63ab3
1662945,1147.600000000,penalty,SLASHING PENALTY NZ,f8513815-ad09-b742-fef8-9814f2313b80
1662946,1147.600000000,penaltydrawn,SLASHING PENALTY DRAWN NZ,12687ef0-6348-9e50-7885-e5292a10eda7
1662947,1147.600000000,shot,OUTSIDE SHOT FOR ONNET,3cfc8139-e81a-8949-e3fc-0a6728b63ab3
1662948,1148.970000000,goal,GOAL,3cfc8139-e81a-8949-e3fc-0a6728b63ab3


In [34]:
print("\n=== 6.4: Re-evaluate Distribution After Reordering ===\n")

is_goal = df_events['event_type'] == 'goal'
total_goals = is_goal.sum()

# Find new preceding event types after reordering
new_preceding = df_events.groupby(['game_id', 'sequence_id'])['event_type'].shift(1)

# Get distribution
new_preceding_dist = new_preceding[is_goal].value_counts()
print(f"AFTER REORDERING - Distribution of events immediately before {total_goals:,} goals:")
for event_type, count in new_preceding_dist.items():
    pct = (count / total_goals) * 100
    print(f"  {event_type:20s}: {count:5,} ({pct:5.1f}%)")

# Check if there are still problematic cases (non-standard preceding events)
still_problematic = is_goal & ~new_preceding.isin(['shot', 'deflection', 'block', 'whistle', 'faceoff'])
still_count = still_problematic.sum()

if still_count > 0:
    print(f"\n⚠ Still {still_count:,} goals preceded by non-standard events")
else:
    print("\n✓ All goals properly preceded by standard events")

# Record rows before filtering faceoffs
rows_before_faceoff_drop = len(df_events)

# Drop faceoff events missing player_id across all games before inspection
faceoff_mask_all = (df_events['event_type'] == 'faceoff') & (df_events['player_id'].isna())
faceoffs_removed = faceoff_mask_all.sum()
if faceoffs_removed > 0:
    df_events = df_events[~faceoff_mask_all]
    print(f"Removed {faceoffs_removed:,} faceoff rows missing player_id (rows: {rows_before_faceoff_drop:,} → {len(df_events):,})")
else:
    print("No faceoff rows with missing player_id found")

print(f"\n=== Extract First Game for Inspection ===\n")

# Reset index after faceoff drop
df_events = df_events.reset_index(drop=True)
first_game_id = df_events['game_id'].iloc[0]
first_game = df_events[df_events['game_id'] == first_game_id].copy()

print(f"Game ID: {first_game_id}")
print(f"Events: {len(first_game):,}")
print(f"Sequences: {first_game.groupby('sequence_id').ngroups:,}")
print(f"Goals in first game: {(first_game['event_type'] == 'goal').sum():,}")

# Export to CSV
output_csv = OUTPUT_DIR / "first_game_inspection.csv"
first_game.to_csv(output_csv, index=False, float_format='%.6f')
print(f"✓ Exported to {output_csv.name}")

print("\n=== Section 6 Complete ===")
print("✓ Assists removed")
print("✓ Events before goals evaluated")
print("✓ Events reordered")
print("✓ Distribution re-evaluated")
print("✓ Faceoff artifacts removed")


=== 6.4: Re-evaluate Distribution After Reordering ===

AFTER REORDERING - Distribution of events immediately before 2,810 goals:
  shot                : 2,367 ( 84.2%)
  deflection          :   259 (  9.2%)
  block               :   184 (  6.5%)

✓ All goals properly preceded by standard events
Removed 28,095 faceoff rows missing player_id (rows: 1,686,507 → 1,658,412)

=== Extract First Game for Inspection ===

Game ID: 00b0366a-95c6-5250-2dae-e3dd5c4198bc
Events: 3,261
Sequences: 52
Goals in first game: 6
✓ Exported to first_game_inspection.csv

=== Section 6 Complete ===
✓ Assists removed
✓ Events before goals evaluated
✓ Events reordered
✓ Distribution re-evaluated
✓ Faceoff artifacts removed


## 7. Score Verification

Verify that our cleaned event data produces correct game scores by:
1. Loading official game scores from `games.parquet`
2. Counting goals in the final cleaned events (from `events_clean_final.parquet`)
3. Adding penalty shot and shootout goals (from `extra_goals_for_verification.csv`)
4. Comparing calculated scores to official scores

In [35]:
print("=" * 60)
print("SCORE VERIFICATION")
print("=" * 60)

# Load official game scores
print("\n1. Loading official game scores...")
df_games = pd.read_parquet(DATA_DIR / "games.parquet")
print(f"   ✓ Loaded {len(df_games)} games")

event_goals = df_events[df_events['event_type'] == 'goal'].groupby(['game_id', 'team_id']).size().reset_index(name='event_goals')
print(f"   ✓ Found {event_goals['event_goals'].sum()} goals in events")

# Load extra goals (penalty shots + shootouts)
print("\n3. Loading extra goals (penalty shots + shootouts)...")
df_extra = pd.read_csv(OUTPUT_DIR / "extra_goals_for_verification.csv")
# Each row = 1 goal (penalty shots are real goals, shootout winners get 1 goal in box score)
extra_goals = df_extra.groupby(['game_id', 'team_id']).size().reset_index(name='extra_goals')
print(f"   ✓ Found {len(df_extra)} extra goals (each worth 1 goal in box score)")
print(f"     - Penalty shot goals: {len(df_extra[df_extra['goal_type'] == 'penalty_shot'])}")
print(f"     - Shootout winners: {len(df_extra[df_extra['goal_type'] == 'shootout_winner'])}")

# Merge event goals and extra goals
print("\n4. Calculating total goals per team per game...")
goals_combined = event_goals.merge(extra_goals, on=['game_id', 'team_id'], how='outer').fillna(0)
goals_combined['total_goals'] = goals_combined['event_goals'] + goals_combined['extra_goals']
goals_combined[['event_goals', 'extra_goals', 'total_goals']] = goals_combined[['event_goals', 'extra_goals', 'total_goals']].astype(int)

# Create wide format for comparison (one row per game)
goals_wide = goals_combined.pivot(index='game_id', columns='team_id', values='total_goals').reset_index()
goals_wide.columns.name = None  # Remove the columns name

# Prepare official scores for comparison
# Assuming games has columns like: home_team_id, away_team_id, home_score, away_score
# Create a comparable format
official_scores = df_games[['game_id', 'home_team_id', 'away_team_id', 'home_score', 'away_score']].copy()

# Merge and compare
comparison = official_scores.merge(goals_wide, on='game_id', how='left')

# Check for mismatches
print("\n5. Comparing calculated vs official scores...")
mismatches = []

for idx, row in comparison.iterrows():
    home_team = row['home_team_id']
    away_team = row['away_team_id']
    
    # Handle NaN (team with 0 goals won't appear in pivot)
    calc_home = row.get(home_team, 0)
    calc_away = row.get(away_team, 0)
    calc_home = 0 if pd.isna(calc_home) else int(calc_home)
    calc_away = 0 if pd.isna(calc_away) else int(calc_away)
    
    official_home = row['home_score']
    official_away = row['away_score']
    
    if calc_home != official_home or calc_away != official_away:
        mismatches.append({
            'game_id': row['game_id'],
            'home_team': home_team,
            'away_team': away_team,
            'calc_home': calc_home,
            'calc_away': calc_away,
            'official_home': official_home,
            'official_away': official_away,
            'home_diff': calc_home - official_home,
            'away_diff': calc_away - official_away
        })

if len(mismatches) == 0:
    print(f"   ✓ SUCCESS! All {len(df_games)} game scores match!")
else:
    print(f"   ✗ MISMATCHES FOUND: {len(mismatches)} games have score discrepancies")
    print("\n   Details:")
    df_mismatches = pd.DataFrame(mismatches)
    print(df_mismatches.to_string(index=False))
    print(f"\n   Total goals off: {df_mismatches['home_diff'].abs().sum() + df_mismatches['away_diff'].abs().sum()}")

print("\n" + "=" * 60)
print("Score verification complete!")
print("=" * 60)

SCORE VERIFICATION

1. Loading official game scores...
   ✓ Loaded 480 games
   ✓ Found 2810 goals in events

3. Loading extra goals (penalty shots + shootouts)...
   ✓ Found 41 extra goals (each worth 1 goal in box score)
     - Penalty shot goals: 6
     - Shootout winners: 35

4. Calculating total goals per team per game...

5. Comparing calculated vs official scores...
   ✓ SUCCESS! All 480 game scores match!

Score verification complete!


## 8. Final EDA & Export

Summarize the cleaned dataset after faceoff removal, verify the row count change, and export `events_clean_final.parquet` for downstream use.

In [36]:
print("=== Final EDA and Export ===")

df_final_clean = df_events.copy()
rows_final = len(df_final_clean)
rows_removed = rows_before_faceoff_drop - rows_final
print(f"Rows before faceoff removal: {rows_before_faceoff_drop:,}")
print(f"Rows after faceoff removal: {rows_final:,}")
print(f"Faceoff rows removed (should match tracked count {faceoffs_removed:,}): {rows_removed:,}")

num_games = df_final_clean['game_id'].nunique()
num_sequences = df_final_clean.groupby(['game_id', 'sequence_id']).ngroups
num_goals = (df_final_clean['event_type'] == 'goal').sum()
print(f"\nGame/Sequence Stats: {num_games:,} games, {num_sequences:,} sequences, {num_goals:,} goals")

seq_lengths = df_final_clean.groupby(['game_id', 'sequence_id']).size()
print(f"\nSequence length → mean: {seq_lengths.mean():.2f}, median: {seq_lengths.median():.0f}, 95th pct: {seq_lengths.quantile(0.95):.0f}, max: {seq_lengths.max():,}")

seq_outcomes = df_final_clean.groupby(['game_id', 'sequence_id'])['event_type'].last()
goal_seq_rate = (seq_outcomes == 'goal').mean() * 100
print(f"\nGoal sequences: {goal_seq_rate:.2f}%")

key_cols = ['game_id', 'sequence_id', 'event_type', 'team_id', 'period', 'period_time']
missing = df_final_clean[key_cols].isna().sum()
if missing.sum() == 0:
    print("\nNo missing values in core columns")
else:
    print("\nMissing values detected:")
    print(missing[missing > 0])

output_path_final = OUTPUT_DIR / "events_clean_final.parquet"
df_final_clean.to_parquet(output_path_final, index=False)
print(f"\nSaved final cleaned file to {output_path_final.name} (columns: {len(df_final_clean.columns)})")
print("\n=== Final EDA complete ===")

=== Final EDA and Export ===
Rows before faceoff removal: 1,686,507
Rows after faceoff removal: 1,658,412
Faceoff rows removed (should match tracked count 28,095): 28,095

Game/Sequence Stats: 480 games, 28,095 sequences, 2,810 goals

Sequence length → mean: 59.03, median: 45, 95th pct: 158, max: 508

Goal sequences: 10.00%

Missing values detected:
team_id    25254
dtype: int64

Saved final cleaned file to events_clean_final.parquet (columns: 26)

=== Final EDA complete ===


## Phase 1 Summary & Next Steps

### Phase 1 (COMPLETE) ✓
- Removed shootout and penalty-shot sequences, metadata events, and penalty bookkeeping events so every sequence ends with a whistle or goal.
- Vectorized merging of passes/outcomes, false-start faceoff removal, and duplication checks to clean the event-level table without row-by-row loops.
- Enforced causal ordering by inspecting events immediately before goals, pushing LPR/penalty types behind their shot/deflection anchors, and re-validating EOS token coverage.
- Generated diagnostics for penalty-context goals, cleaned assist leakage, trimmed faceoffs missing `player_id`, plus the per-game inspection slice.
- Verified every game score: counted goals from the cleaned events, appended penalty-shot/shootout winners, and confirmed official scores match across 480 games.
- Final EDA confirms dataset size, sequence counts, goal rates, and missing-value sanity checks before exporting `events_clean_final.parquet` for downstream work.

### Phase 2 Plan (Tensor-Ready Post-Processing)
1. **Tensor preparation start**: load `events_clean_final.parquet`, build vocabularies for categorical columns, encode categories, and pad/truncate sequences for batching, keeping the tensor-ready section in the new Phase 2 notebook.
2. **Stints merge**: align each event with the stints data to add `n_home_skaters`, `n_away_skaters`, `score_differential`, `is_power_play_home`, and empty-net flags so the model knows the skater context described in the README.
3. **Tracking integration**: round tracking timestamps, merge on `game_id`/`period`/`match_time`, rotate/translate coordinates so the puck is origin, and construct the 24-dimensional relative player-position snapshot that feeds the transformer’s tracking branch.

### Design Alignment (README Goals)
- The event table now obeys the Phase 1 data-cleaning checklist: no leakage, consistent EOS markers, cleaned passes, and drop of assist/shootout noise as outlined in the README.
- Phase 2 will follow the outlined architecture: transformer encoder with separate event and tracking projections, raw continuous features, and short sequences (≤100 events) that respect the README’s causal sorting and EOS policies.
- Keeping tracking as a "sidecar" 24-dimensional vector lets the model learn pressure/spacing without manual heuristics, matching the README’s physics-vs-rules philosophy (transformer predictions plus static penalty ledger adjustments will come after Phase 2).